# 3. Run PyNNLF: AEDP Aggregation Levels

Prepared for manual execution. Runs `ds25`-`ds36` for 1-day-ahead forecasting using naive, linear regression, and XGBoost. These are the 12 datasets produced by notebook `2.2` after the revision to 3 random samples per aggregation level.


## 1. Setup And Batch Spec

In [ ]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
import yaml
sys.path.insert(0, str(REPO_ROOT / "src"))
import pynnlf  # noqa: E402
BATCH_PATH = PROJECT_DIR / "specs" / "aedp_aggregation_fh8_batch.yaml"
RESULTS_ROOT = PROJECT_DIR / "experiment_result"
TEMP_SPEC_PATH = PROJECT_DIR / "specs" / "_tmp_aedp_aggregation_single.yaml"
batch = yaml.safe_load(BATCH_PATH.read_text(encoding="utf-8"))
config = yaml.safe_load((PROJECT_DIR / "specs" / "pynnlf_config.yaml").read_text(encoding="utf-8"))
forecast_horizon_minutes = {key: int(value) for key, value in config["forecast_horizons"].items()}
print(batch)

## 2. Validate All Input Datasets Exist

In [ ]:
missing = []
for dataset_id in batch["datasets"]:
    matches = sorted((PROJECT_DIR / "data").glob(f"{dataset_id}_*.csv"))
    if len(matches) != 1:
        missing.append((dataset_id, len(matches)))
if missing:
    raise FileNotFoundError("Missing or ambiguous AEDP aggregation datasets. Run notebook 2.2 first. Problem dataset IDs: " + str(missing[:10]))
print(f"Validated {len(batch['datasets'])} dataset files: {batch['datasets'][0]} through {batch['datasets'][-1]}")

## 3. Run Missing Experiments

This cell runs PyNNLF and is resumable.

In [ ]:
def completed_keys(results_root):
    keys = set()
    for result_file in sorted(results_root.glob("E*/E*_a1_experiment_result.csv")):
        try:
            row = pd.read_csv(result_file, nrows=1).iloc[0]
            keys.add((str(row.get("dataset_no", "")), int(row.get("forecast_horizon_min")), str(row.get("model_no", "")), str(row.get("hyperparameter_no", ""))))
        except Exception:
            continue
    return keys
done = completed_keys(RESULTS_ROOT)
total = len(batch["datasets"]) * len(batch["forecast_horizons"]) * len(batch["model_and_hp"])
run_index = 0
for dataset_id in batch["datasets"]:
    for forecast_horizon_id in batch["forecast_horizons"]:
        horizon_minutes = forecast_horizon_minutes[forecast_horizon_id]
        for model_id, hp in batch["model_and_hp"]:
            run_index += 1
            key = (str(dataset_id), horizon_minutes, str(model_id), str(hp))
            if key in done:
                print(f"[skip {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
                continue
            print(f"[run {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
            temp_spec = {"datasets": [dataset_id], "forecast_horizons": [forecast_horizon_id], "model_and_hp": [[model_id, hp]]}
            TEMP_SPEC_PATH.write_text(yaml.safe_dump(temp_spec, sort_keys=False), encoding="utf-8")
            pynnlf.run_experiment_batch(TEMP_SPEC_PATH, plot_enabled=False)
            done.add(key)
if TEMP_SPEC_PATH.exists():
    TEMP_SPEC_PATH.unlink()
pynnlf.recap_experiments(RESULTS_ROOT)
print(f"Recap written: {RESULTS_ROOT / 'a1_experiment_result.csv'}")